<a href="https://colab.research.google.com/github/Arceus2006/AIML-project/blob/main/Support_Vector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# SUPPORT VECTOR CLASSIFIER (SVC)
# HOUSE PRICE DATASET
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    roc_auc_score
)


# ============================================
# 1. LOAD DATA
# ============================================

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)


# ============================================
# 2. CREATE CLASS LABELS
# ============================================

def price_category(price):

    if price < 150000:
        return "Low"

    elif price < 300000:
        return "Medium"

    else:
        return "High"


train["PriceCategory"] = train["SalePrice"].apply(
    price_category
)


print("\nClass distribution:")
print(train["PriceCategory"].value_counts())


# ============================================
# 3. SEPARATE FEATURES AND TARGET
# ============================================

X = train.drop(
    ["SalePrice", "PriceCategory"],
    axis=1
)

y = train["PriceCategory"]


# ============================================
# 4. IDENTIFY COLUMN TYPES
# ============================================

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X.select_dtypes(
    include=["object"]
).columns


print("\nNumerical features:",
      len(numerical_features))

print("Categorical features:",
      len(categorical_features))


# ============================================
# 5. TRAIN-TEST SPLIT
# ============================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================
# 6. NUMERICAL PREPROCESSING
# ============================================

numerical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    # SVM works better when numerical features
    # are on a similar scale
    (
        "scaler",
        StandardScaler()
    )
])


# ============================================
# 7. CATEGORICAL PREPROCESSING
# ============================================

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),

    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])


# ============================================
# 8. COMBINE PREPROCESSING
# ============================================

preprocessor = ColumnTransformer([
    (
        "numerical",
        numerical_pipeline,
        numerical_features
    ),

    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])


# ============================================
# 9. SUPPORT VECTOR CLASSIFIER
# ============================================

classifier = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale"
)


# ============================================
# 10. COMPLETE PIPELINE
# ============================================

model = Pipeline([
    (
        "preprocessing",
        preprocessor
    ),

    (
        "classifier",
        classifier
    )
])


# ============================================
# 11. TRAIN MODEL
# ============================================

print("\nTraining SVC...")

model.fit(
    X_train,
    y_train
)

print("Training completed!")


# ============================================
# 12. PREDICTION
# ============================================

y_pred = model.predict(X_val)


# ============================================
# 13. ACCURACY
# ============================================

accuracy = accuracy_score(
    y_val,
    y_pred
)

print("\n================================")
print("MODEL PERFORMANCE")
print("================================")

print("Accuracy:", accuracy)


# ============================================
# 14. CLASSIFICATION REPORT
# ============================================

print("\nClassification Report:")

print(
    classification_report(
        y_val,
        y_pred
    )
)


# ============================================
# 15. CONFUSION MATRIX
# ============================================

print("\nConfusion Matrix:")

cm = confusion_matrix(
    y_val,
    y_pred
)

print(cm)


# ============================================
# 16. ACTUAL VS PREDICTED
# ============================================

results = pd.DataFrame({
    "Actual": y_val.values,
    "Predicted": y_pred
})

print("\nActual vs Predicted:")
print(results.head(20))

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# ============================================
# MULTICLASS ROC CURVE
# ============================================

classes = model.classes_

# Convert actual labels to binary form
y_test_binary = label_binarize(
    y_val,
    classes=classes
)

# Get probabilities
y_probability = model.predict_proba(X_val)


plt.figure(figsize=(8, 6))

for i, class_name in enumerate(classes):

    fpr, tpr, _ = roc_curve(
        y_test_binary[:, i],
        y_probability[:, i]
    )

    auc_score = auc(
        fpr,
        tpr
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{class_name} (AUC = {auc_score:.2f})"
    )


# Random classifier line
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("Multiclass ROC Curve")

plt.legend()

plt.show()

Train shape: (1460, 81)
Test shape: (1459, 80)

Class distribution:
PriceCategory
Medium    730
Low       615
High      115
Name: count, dtype: int64

Numerical features: 37
Categorical features: 43

Training SVC...
Training completed!

MODEL PERFORMANCE
Accuracy: 0.8938356164383562

Classification Report:
              precision    recall  f1-score   support

        High       0.89      0.74      0.81        23
         Low       0.90      0.92      0.91       123
      Medium       0.89      0.90      0.89       146

    accuracy                           0.89       292
   macro avg       0.89      0.85      0.87       292
weighted avg       0.89      0.89      0.89       292


Confusion Matrix:
[[ 17   0   6]
 [  0 113  10]
 [  2  13 131]]

Actual vs Predicted:
    Actual Predicted
0      Low       Low
1      Low       Low
2   Medium    Medium
3   Medium    Medium
4   Medium    Medium
5      Low       Low
6      Low       Low
7      Low       Low
8     High      High
9   Medium    

AttributeError: This 'Pipeline' has no attribute 'predict_proba'